# Intel Image Classification

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
batch_size = 128
lr = 1e-3
epochs = 50

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

In [ ]:
train_data_dir = os.path.join(path, "seg_train", "seg_train")
test_data_dir = os.path.join(path, "seg_test", "seg_test")
print(train_data_dir)
print(test_data_dir)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(root = train_data_dir, transform = train_transform)
test_dataset = datasets.ImageFolder(root = test_data_dir, transform = test_transform)

train_loader = DataLoader(dataset = train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(dataset = test_dataset, batch_size = batch_size, shuffle = False)

classes = train_dataset.classes

print(f"Train = {len(train_dataset)}, Test = {len(test_dataset)}")
print(f"Intel Image Classification classes: {classes}")

In [ ]:
x, t = next(iter(train_loader))
x = x[0].numpy().transpose(1, 2, 0)
t = t[0].numpy()

print(f"Class = {classes[t]}")
plt.imshow(x)
plt.show()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.name = "SimpleCNN"
        
        self.conv1 = nn.Conv2d( 3, 32, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
        self.pool1 = nn.MaxPool2d(kernel_size = 2)
        self.pool2 = nn.MaxPool2d(kernel_size = 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fclsv = nn.Linear(128, 2)
        self.fcout = nn.Linear(2, 6)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        v = self.fclsv(x)
        y = self.fcout(v)

        return y, v

In [ ]:
model = CNN()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
history = {"loss" : [], "accuracy" : [], "test_loss" : [], "test_accuracy" : []}

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_accuracy = 0

    for x, t in tqdm(train_loader):
        x, t = x.to(device), t.to(device)

        y, v = model(x)
        loss = criterion(y, t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        train_accuracy += (y.argmax(1) == t).sum().item()

    train_loss /= len(train_dataset)
    train_accuracy /= len(train_dataset)

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)

    model.eval()
    test_loss = 0
    test_accuracy = 0

    with torch.no_grad():
        for x, t in tqdm(test_loader):
            x, t = x.to(device), t.to(device)
            y, v = model(x)
            loss = criterion(y, t)
            test_loss += loss.item() * x.size(0)
            test_accuracy += (y.argmax(1) == t).sum().item()

    test_loss /= len(test_dataset)
    test_accuracy /= len(test_dataset)

    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(f"Epoch = {epoch + 1}, ", end = "")
    print(f"Loss = {train_loss:.5f}, Accuracy = {train_accuracy:.5f}, ", end = "")
    print(f"Test Loss = {test_loss:.5f}, Test Accuracy = {test_accuracy:.5f}")

torch.save({
    "model" : model.state_dict(),
    "optimizer" : optimizer.state_dict(),
    "history" : history
}, f"{model.name}.pth")

In [ ]:
plt.plot(range(1, epochs + 1), history["loss"], label = "Train")
plt.plot(range(1, epochs + 1), history["test_loss"], label = "Test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle = "--")
plt.legend()
plt.show()

In [ ]:
plt.plot(range(1, epochs + 1), history["accuracy"], label = "Train")
plt.plot(range(1, epochs + 1), history["test_accuracy"], label = "Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle = "--")
plt.legend()
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    test_accuracy = 0

    class_accuracy = [0.0 for i in range(len(classes))]
    class_datas = [0.0 for i in range(len(classes))]

    for x, t in tqdm(test_loader):
        x, t = x.to(device), t.to(device)
        y, v = model(x)
        tmp = y.argmax(1) == t
        test_accuracy += tmp.sum().item()

        c = tmp.squeeze()

        for i in range(len(t)):
            label = t[i].item()
            class_accuracy[label] += c[i].item()
            class_datas[label] += 1

    test_accuracy /= len(test_dataset)
    print(f"Test Accuracy = {test_accuracy:.5f}")

    for i in range(len(classes)):
        class_accuracy[i] /= class_datas[i]
        print(f"Accuracy of {classes[i]} = {class_accuracy[i]:.5f}")

In [ ]:
plt.bar(classes, class_accuracy)
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.xticks(rotation = 45)
plt.ylim(0.0, 1.0)
plt.grid(axis = "y", linestyle = "--")
plt.show()

In [ ]:
vs = []
ts = []

count = {i : 0 for i in range(len(classes))}
datas = 100

shuffled_test_dataset = DataLoader(test_dataset, batch_size = 1, shuffle = True)

with torch.no_grad():
    for x, t in shuffled_test_dataset:
        label = t.item()

        if count[label] < datas:
            x = x.to(device)
            y, v = model(x)
            vs.append(v.cpu().numpy().squeeze())
            ts.append(label)
            count[label] += 1

        if all(c == datas for c in count.values()):
            break

vs = np.array(vs)
ts = np.array(ts)

print(f"Total Datas = {len(vs)}")

In [ ]:
for i, label in enumerate(classes):
    class_idx = np.where(ts == i)
    class_v = vs[class_idx]

    plt.scatter(class_v[:, 0], class_v[:, 1], label = label)

plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.grid(True, linestyle = "--")
plt.legend()
plt.show()